# Treino YOLO11 - Deteccao de EPI (Colab) - v2 (gloves/goggles reforcados, sem person)

Antes de rodar:
1. Suba o arquivo `yolo_v2.zip` para o seu Google Drive, na pasta `MyDrive/TCC/yolo_v2.zip`.
2. Em `Runtime > Change runtime type`, selecione GPU.
3. Rode as celulas abaixo em ordem.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/TCC/yolo_v2.zip'
DATA_DIR = '/content/yolo'

!mkdir -p {DATA_DIR}
!unzip -q -o "{ZIP_PATH}" -d /content

# Precaucao de permissao (o zip da v1 vinha do Windows com permissoes zeradas;
# este foi gerado no Linux, mas o chmod nao faz mal nenhum manter).
!chmod -R u+rwX {DATA_DIR}
!ls -la {DATA_DIR}

In [ ]:
# Checagem antes de gastar GPU: confirma que os arquivos estao realmente visiveis
import os

for split in ['train', 'val', 'test']:
    img_dir = f'{DATA_DIR}/{split}/images'
    lbl_dir = f'{DATA_DIR}/{split}/labels'
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f'{split}: {n_img} imagens, {n_lbl} labels')
    assert n_img > 0, f'ERRO: {img_dir} esta vazio ou inacessivel!'
    assert n_img == n_lbl, f'ERRO: numero de imagens e labels difere em {split}!'

print('OK - dataset pronto para treino')

In [ ]:
%pip install -q ultralytics
!yolo settings sync=False
import ultralytics
ultralytics.checks()

In [ ]:
# Reescreve data.yaml com caminhos absolutos do Colab
data_yaml = f"""train: {DATA_DIR}/train/images
val: {DATA_DIR}/val/images
test: {DATA_DIR}/test/images
nc: 5
names: ['helmet', 'vest', 'gloves', 'boots', 'goggles']
"""

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(data_yaml)

print(data_yaml)

## Treino

Mesmos hiperparametros usados localmente (`runs/detect/train-3/args.yaml`). Os resultados sao salvos direto no Drive (`project=`) para nao se perderem se a sessao do Colab cair.

In [ ]:
!mkdir -p /content/drive/MyDrive/TCC/runs

!yolo task=detect mode=train \
  model=yolo11n.pt \
  data={DATA_DIR}/data.yaml \
  epochs=100 \
  batch=16 \
  imgsz=640 \
  project=/content/drive/MyDrive/TCC/runs \
  name=train-colab-v2

## Validacao

In [ ]:
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v2/weights/best.pt \
  data={DATA_DIR}/data.yaml